# Qdrant Indexing - IMDB Movies

Este notebook maneja la indexacion de peliculas en Qdrant para busqueda hibrida (embeddings + BM25).

**Ejecutar este notebook cuando:**
- Se inicializa Qdrant por primera vez
- Se actualiza la base de datos de peliculas
- Se necesita recrear el indice

**Para busquedas/consultas:** usar `serving.ipynb`

In [ ]:
import polars as pl
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

## 1. Configuracion

In [ ]:
# ============= CONFIGURACION =============
import os

# Modelo de embeddings
EMBEDDINGS_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

# Qdrant - use environment variable or default to localhost
# Set QDRANT_URL environment variable for remote:
#   export QDRANT_URL="http://<external-ip>:6333"
# Or for port-forward:
#   kubectl port-forward svc/qdrant 6333:6333
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION_NAME = "imdb-movies-hybrid"

# Indexing
BATCH_SIZE = 500
MOVIES_PARQUET_PATH = "data/complete_imdb_database.parquet"

print(f"Qdrant URL: {QDRANT_URL}")

## 2. Conexion a Qdrant

In [ ]:
# Conectar a Qdrant
client = QdrantClient(QDRANT_URL)

# Verificar colecciones existentes
collections = [c.name for c in client.get_collections().collections]
print(f"Colecciones existentes: {collections}")

## 3. Crear Coleccion

In [ ]:
# Crear coleccion con soporte para busqueda hibrida (dense + sparse/BM25)
if COLLECTION_NAME not in collections:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(
                size=EMBEDDING_DIM,
                distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(
                modifier=models.Modifier.IDF,  # BM25-like scoring
            )
        }
    )
    print(f"Coleccion '{COLLECTION_NAME}' creada con vectores densos y sparse (BM25)")
else:
    print(f"Coleccion '{COLLECTION_NAME}' ya existe")
    
# Info de la coleccion
collection_info = client.get_collection(COLLECTION_NAME)
print(f"Puntos en coleccion: {collection_info.points_count}")

## 4. Cargar Datos

In [ ]:
# Cargar base de datos de peliculas
movies_db = pl.read_parquet(MOVIES_PARQUET_PATH)

# Agregar features derivadas
if "imdb_votes_log" not in movies_db.columns:
    movies_db = movies_db.with_columns([
        pl.col("imdb_votes").log1p().alias("imdb_votes_log"),
    ])

# Crear texto completo para embeddings
movies_db = movies_db.with_columns([
    pl.concat_str(
        [pl.col("title"), pl.col("Plot")],
        separator=". ",
    ).fill_null("").alias("full_text"),
])

print(f"Peliculas cargadas: {movies_db.height}")
movies_db.select(["imdb_id", "title", "genres", "imdb_rating", "full_text"]).head(3)

## 5. Funciones de Indexacion

In [ ]:
def tokenize_text(text: str) -> dict[int, float]:
    """Tokenizar texto para sparse vectors (BM25-like)."""
    if not text:
        return {}
    
    # Tokenizacion simple: palabras en minusculas
    words = text.lower().split()
    # Crear indices basados en hash de palabras
    token_counts = {}
    for word in words:
        # Filtrar palabras muy cortas
        if len(word) > 2:
            idx = hash(word) % 100000  # Limitar indices
            token_counts[idx] = token_counts.get(idx, 0) + 1
    
    return token_counts


def index_movies_to_qdrant(
    movies_df: pl.DataFrame, 
    client: QdrantClient, 
    collection_name: str, 
    embedding_model: SentenceTransformer,
    batch_size: int = 100,
    force_reindex: bool = False
):
    """Indexar peliculas en Qdrant con vectores densos y sparse."""
    
    collection_info = client.get_collection(collection_name)
    if collection_info.points_count > 0 and not force_reindex:
        print(f"Coleccion ya tiene {collection_info.points_count} puntos. Saltando indexacion.")
        print("Usa force_reindex=True para re-indexar.")
        return
    
    if force_reindex and collection_info.points_count > 0:
        print(f"Eliminando {collection_info.points_count} puntos existentes...")
        client.delete(
            collection_name=collection_name,
            points_selector=models.FilterSelector(
                filter=models.Filter(must=[])
            )
        )
    
    print(f"Indexando {movies_df.height} peliculas...")
    
    texts = movies_df["full_text"].to_list()
    imdb_ids = movies_df["imdb_id"].to_list()
    
    # Generar embeddings en batches
    total_batches = (len(texts) + batch_size - 1) // batch_size
    
    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(texts))
        
        batch_texts = texts[start:end]
        batch_ids = imdb_ids[start:end]
        
        # Generar embeddings densos
        dense_embeddings = embedding_model.encode(batch_texts, show_progress_bar=False)
        
        # Crear puntos
        points = []
        for i, (text, imdb_id) in enumerate(zip(batch_texts, batch_ids)):
            # Sparse vector (BM25-like)
            sparse_tokens = tokenize_text(text)
            
            point = models.PointStruct(
                id=start + i,  # ID numerico
                vector={
                    "dense": dense_embeddings[i].tolist(),
                    "bm25": models.SparseVector(
                        indices=list(sparse_tokens.keys()),
                        values=list(sparse_tokens.values())
                    )
                },
                payload={
                    "imdb_id": imdb_id,
                    "row_index": start + i,
                }
            )
            points.append(point)
        
        # Upsert batch
        client.upsert(collection_name=collection_name, points=points)
        
        if (batch_idx + 1) % 50 == 0 or batch_idx == total_batches - 1:
            print(f"  Batch {batch_idx + 1}/{total_batches} completado")
    
    print(f"Indexacion completada. Total puntos: {client.get_collection(collection_name).points_count}")

## 6. Ejecutar Indexacion

In [ ]:
# Cargar modelo de embeddings
print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer(EMBEDDINGS_MODEL)

# Indexar peliculas
index_movies_to_qdrant(
    movies_db, 
    client, 
    COLLECTION_NAME, 
    embedding_model, 
    batch_size=BATCH_SIZE,
    force_reindex=False  # Cambiar a True para re-indexar
)

## 7. Verificar Indexacion

In [ ]:
# Verificar la coleccion
collection_info = client.get_collection(COLLECTION_NAME)

print(f"Coleccion: {COLLECTION_NAME}")
print(f"Total puntos: {collection_info.points_count}")
print(f"Vectores config: {collection_info.config.params.vectors}")
print(f"Sparse vectors config: {collection_info.config.params.sparse_vectors}")

In [ ]:
# Test rapido de busqueda
test_query = "time travel science fiction"
query_embedding = embedding_model.encode([test_query])[0].tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    using="dense",
    limit=5,
    with_payload=True,
)

print(f"Test query: '{test_query}'")
print(f"Resultados: {len(results.points)}")
for hit in results.points:
    imdb_id = hit.payload["imdb_id"]
    movie = movies_db.filter(pl.col("imdb_id") == imdb_id)
    if not movie.is_empty():
        print(f"  - {movie['title'][0]} (score: {hit.score:.4f})")

## 8. Utilidades (Opcional)

In [ ]:
# PELIGRO: Descomentar solo si necesitas eliminar la coleccion
# client.delete_collection(COLLECTION_NAME)
# print(f"Coleccion '{COLLECTION_NAME}' eliminada")